# Train Level 10 (Single-Level PPO)\n\nThis notebook is generated from train_colab_single_levels.ipynb and runs a single level end-to-end.\n\nTarget level: normal-10\n

# Bobby Carrot - Per-Level PPO (L1-L10 Demo Runs)

Goal: train one specialized PPO policy per normal level (1-10) for robust single-level demonstrations.

This notebook is independent of train_colab.ipynb (the phased L1-25 curriculum pipeline).

## Algorithm

PPO + optional ICM curiosity.
All curriculum/anti-forgetting/EMA-teacher machinery is disabled (one level = one policy).

## Tier summary (PPO presets)

| Tier   | Levels  | Mechanics                          | Budget  | PPO lr | rollout | entropy (start->min) | gamma | ICM |
|--------|---------|------------------------------------|---------|--------|---------|----------------------|-------|-----|
| A1     | L1      | floor + carrot                     | 150k    | 2e-4   | 1024    | 0.15 -> 0.05         | 0.99  | off |
| A2     | L2-L3   | + crumble-intro (one-use bridges)  | 500k    | 1.5e-4 | 2048    | 0.20 -> 0.08         | 0.995 | on  |
| B      | L4-L7   | + crumble chains + arrows          | 400k    | 1e-4   | 2048    | 0.15 -> 0.08         | 0.995 | on  |
| C      | L8-L10  | + conveyor belts                   | 600k    | 8e-5   | 2048    | 0.15 -> 0.08         | 0.995 | on  |

## Reliability target

For L2/L3, training now uses a greedy stability gate:
- periodic greedy eval windows
- threshold: >=95% success
- requirement: 10 consecutive passing windows
- each window evaluates 20 episodes

This prevents selecting checkpoints that look good only under stochastic sampling.

---
## 1. Setup — Clone, install, mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO = '/content/Mini_Project_Game'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=False)
    print('Repo updated (git pull)')
else:
    subprocess.run(['git', 'clone', 'https://github.com/Charan20510/Mini_Project_Game.git', REPO], check=True)
    print('Repo cloned')
%cd /content/Mini_Project_Game

In [ ]:
%pip install torch numpy pandas matplotlib pillow --quiet

In [ ]:
import os, sys
os.chdir('/content/Mini_Project_Game')
for p in ('/content/Mini_Project_Game', '/content/Mini_Project_Game/Game_Python'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Working dir:', os.getcwd())

In [ ]:
# Drop stale .pyc so Colab always picks up the latest source after git pull.
import subprocess, importlib
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '*.pyc', '-delete'], check=False)
subprocess.run(['find', '/content/Mini_Project_Game', '-name', '__pycache__', '-type', 'd',
                '-exec', 'rm', '-rf', '{}', '+'], check=False)
importlib.invalidate_caches()
print('pycache cleared')

In [ ]:
# Per-level Drive backup/restore.
DRIVE_ROOT = '/content/drive/MyDrive/Bobby_Carrot_RL/single_level'
os.makedirs(DRIVE_ROOT, exist_ok=True)

def restore_level(level_num: int) -> bool:
    """Copy checkpoints_level{N} from Drive if present. Returns True if restored."""
    import shutil
    src = f'{DRIVE_ROOT}/checkpoints_level{level_num}'
    dst = f'/content/Mini_Project_Game/checkpoints_level{level_num}'
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Restored checkpoints_level{level_num} from Drive')
        return True
    return False

def save_level(level_num: int) -> None:
    """Back up checkpoints_level{N} and logs_level{N} to Drive."""
    import shutil
    for kind in ('checkpoints', 'logs'):
        src = f'/content/Mini_Project_Game/{kind}_level{level_num}'
        dst = f'{DRIVE_ROOT}/{kind}_level{level_num}'
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f'Saved {kind}_level{level_num} to Drive')

print('Drive helpers ready. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# Reload every module touched by training so a `git pull` inside the same
# kernel picks up the latest source — crucial when tuning hyperparameters
# between runs.
import importlib
import Bobby_Carrot.rl_models.config as _cfg
import Bobby_Carrot.rl_models.networks as _nets
import Bobby_Carrot.rl_models.buffers as _bufs
import Bobby_Carrot.rl_models.icm as _icm
import Bobby_Carrot.rl_models.ppo as _ppo
import Bobby_Carrot.rl_models.single_level.level_configs as _lc
import bobby_carrot.rl_env as _env
for m in (_env, _cfg, _nets, _bufs, _icm, _ppo, _lc):
    importlib.reload(m)

from Bobby_Carrot.rl_models.single_level import build_ppo_configs_for_level, LEVEL_TIER
from Bobby_Carrot.rl_models.evaluate import evaluate_agent

SUMMARY = {}  # level_num -> {'success_rate', 'avg_steps', 'checkpoint'}
print('Imports OK. LEVEL_TIER =', LEVEL_TIER)


In [ ]:
# One function called from each per-level cell below.
def train_and_eval_level(level_num: int, episodes: int = 20, save_to_drive: bool = True):
    from Bobby_Carrot.rl_models.single_level.level_configs import build_ppo_configs_for_level
    from Bobby_Carrot.rl_models.ppo import train_ppo
    import pathlib

    # build_ppo_configs_for_level returns (PPOConfig, TrainingConfig, LevelConfig, ICMConfig)
    # with per-tier PPO hyperparameters (rollout_length, entropy_coeff, gamma, ICM, etc.)
    # tuned to fix the L2 oscillation failure mode and ensure >95% success.
    ppo_cfg, train_cfg, level_cfg, icm_cfg = build_ppo_configs_for_level(level_num, checkpoint_root='.')

    train_ppo(ppo_config=ppo_cfg, train_config=train_cfg, level_config=level_cfg, icm_config=icm_cfg)

    # Evaluate the best checkpoint
    root = pathlib.Path('.') / f'checkpoints_level{level_num}' / 'ppo'
    best = root / 'ppo_best.pt'
    final = root / 'ppo_final.pt'
    ckpt = str(best) if best.exists() else str(final)

    print(f'\n--- Evaluating normal-{level_num:02d} ---')
    r = evaluate_agent(
        algo='ppo', checkpoint_path=ckpt,
        levels=[('normal', level_num)],
        episodes_per_level=episodes,
        max_steps=train_cfg.max_steps_per_episode,
        observation_mode='full', device_str='auto',
    )
    agg = r['aggregate']
    SUMMARY[level_num] = {
        'success_rate': agg['success_rate'],
        'avg_steps': agg.get('avg_steps', 0),
        'checkpoint': ckpt,
    }
    print(f"\nLevel {level_num}: success={agg['success_rate']:.1%} | avg_steps={agg.get('avg_steps', 0):.1f}")
    if save_to_drive:
        save_level(level_num)
    return r

print('Helper ready: train_and_eval_level(N)')


In [ ]:
train_and_eval_level(10)\n

In [ ]:
from Bobby_Carrot.rl_models.evaluate import evaluate_agent\nckpt = SUMMARY.get(10, {}).get('checkpoint')\nif ckpt:\n    evaluate_agent(algo='ppo', checkpoint_path=ckpt, levels=[('normal', 10)], episodes_per_level=20, max_steps=500, observation_mode='full', device_str='auto')\nelse:\n    print('No checkpoint recorded in SUMMARY yet.')\n